# Demo — Digital Validation (Path A)

Validates that CRISE saliency correctly identifies the attack surface:
mask the top-k% most salient pixels and measure rank-1 accuracy drop.

**Four conditions:**
1. CRISE-guided (primary)
2. RISE-guided (key comparison — proves CRISE finds better attack surface)
3. Random pixel perturbation (baseline)
4. Bottom-k CRISE pixels (low-saliency control)

**Pixel budgets:** 5%, 10%, 20%, 30% of the 112×112 chip (628–3,763 pixels)

Expected: CRISE-guided curve drops fastest — masking fewer pixels causes
a larger accuracy drop than RISE or random.

In [ ]:
# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

SPLIT_PATH       = "splits/lfw_1N_split.json"
GALLERY_EMB      = "cache/G.npy"
GALLERY_IDS      = "cache/gallery_ids.npy"

CRISE_SAL_DIR    = "results/crise_maps"
RISE_SAL_DIR     = "results/rise_alignedchip_baseline_multi"
CRISE_SUMMARY    = "results/crise_maps/summary_crise_tau0.1_N1000_s8_p0.5_MASTERSEED123.csv"
RISE_SUMMARY     = "results/rise_baseline/summary_K1680_N1000_s8_p0.5_MASTERSEED123.csv"

FIG_DIR          = "results/forensics_figures"

BUDGETS          = [0.05, 0.10, 0.20, 0.30]   # fraction of 112×112 pixels
N_PROBES         = 200    # probes to evaluate (sample from matched pairs)
RANDOM_SEED      = 42

In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------

import os, json
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm
from insightface.app import FaceAnalysis

from rise import build_aligned_chip_112, get_embedding_from_chip

os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
# ---------------------------------------------------------------------------
# InsightFace setup
# ---------------------------------------------------------------------------

app = FaceAnalysis(
    name="buffalo_l",
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
)
app.prepare(ctx_id=0, det_size=(640, 640))
rec = app.models["recognition"]
print("InsightFace ready")

In [ ]:
# ---------------------------------------------------------------------------
# Load gallery + summaries
# ---------------------------------------------------------------------------

G           = np.load(GALLERY_EMB).astype(np.float32)
gallery_ids = np.load(GALLERY_IDS, allow_pickle=True).tolist()
id_to_index = {gid: i for i, gid in enumerate(gallery_ids)}

crise_df = pd.read_csv(CRISE_SUMMARY)
rise_df  = pd.read_csv(RISE_SUMMARY)

# Normalise column names
for df in [crise_df, rise_df]:
    if "true_id" not in df.columns:
        df.rename(columns={df.columns[0]: "true_id"}, inplace=True)

print(f"CRISE summary: {len(crise_df)} rows")
print(f"RISE  summary: {len(rise_df)} rows")

In [ ]:
# ---------------------------------------------------------------------------
# Build matched pairs: probes that have BOTH a CRISE and RISE saliency map
# ---------------------------------------------------------------------------

crise_ok = crise_df[crise_df["saliency_path"].notna() &
                    crise_df["saliency_path"].apply(os.path.exists)].copy()
rise_ok  = rise_df[rise_df["saliency_path"].notna() &
                   rise_df["saliency_path"].apply(os.path.exists)].copy()

# Match on (true_id, probe_path)
merged = crise_ok.merge(
    rise_ok[["true_id", "probe_path", "saliency_path"]],
    on=["true_id", "probe_path"],
    suffixes=("_crise", "_rise"),
)
merged = merged[merged["true_id"].isin(id_to_index)].reset_index(drop=True)

# Sample N_PROBES
rng = np.random.default_rng(RANDOM_SEED)
idx = rng.choice(len(merged), size=min(N_PROBES, len(merged)), replace=False)
pairs = merged.iloc[idx].reset_index(drop=True)

print(f"Matched pairs available: {len(merged)}")
print(f"Evaluating: {len(pairs)} probes")

In [ ]:
# ---------------------------------------------------------------------------
# Perturbation helpers
# ---------------------------------------------------------------------------

CHIP_H, CHIP_W = 112, 112
N_PIXELS = CHIP_H * CHIP_W   # 12544

def mask_top_k(chip_bgr, sal, budget, invert=False):
    """Replace top-k (or bottom-k) salient pixels with black (zeros)."""
    k = int(N_PIXELS * budget)
    flat = sal.ravel()
    order = np.argsort(flat)            # ascending
    indices = order[:k] if invert else order[-k:]   # bottom-k or top-k

    chip_out = chip_bgr.copy().astype(np.uint8)
    ys, xs = np.unravel_index(indices, (CHIP_H, CHIP_W))
    chip_out[ys, xs] = 0   # black — consistent with CRISE saliency computation
    return chip_out


def mask_random(chip_bgr, budget, rng_local):
    """Replace a random set of pixels with black (zeros)."""
    k = int(N_PIXELS * budget)
    indices = rng_local.choice(N_PIXELS, size=k, replace=False)
    chip_out = chip_bgr.copy().astype(np.uint8)
    ys, xs = np.unravel_index(indices, (CHIP_H, CHIP_W))
    chip_out[ys, xs] = 0   # black
    return chip_out


def rank1_correct(chip_bgr, true_id_idx):
    """Return True if ArcFace ranks the true identity first."""
    try:
        emb  = get_embedding_from_chip(chip_bgr, rec)
        sims = G @ emb
        return int(np.argmax(sims)) == true_id_idx
    except Exception:
        return False

In [ ]:
# ---------------------------------------------------------------------------
# Main evaluation loop
# ---------------------------------------------------------------------------

results = {b: {"crise": [], "rise": [], "random": [], "bottom_crise": []}
           for b in BUDGETS}

for idx, row in tqdm(pairs.iterrows(), total=len(pairs), desc="Probes"):
    img = cv2.imread(row["probe_path"])
    if img is None:
        continue

    try:
        chip = build_aligned_chip_112(img, app)
    except Exception:
        continue

    true_idx   = id_to_index[row["true_id"]]
    crise_sal  = np.load(row["saliency_path_crise"]).astype(np.float32)
    rise_sal   = np.load(row["saliency_path_rise"]).astype(np.float32)
    rng_local  = np.random.default_rng(RANDOM_SEED + idx)

    for budget in BUDGETS:
        results[budget]["crise"].append(
            rank1_correct(mask_top_k(chip, crise_sal, budget), true_idx))
        results[budget]["rise"].append(
            rank1_correct(mask_top_k(chip, rise_sal, budget), true_idx))
        results[budget]["random"].append(
            rank1_correct(mask_random(chip, budget, rng_local), true_idx))
        results[budget]["bottom_crise"].append(
            rank1_correct(mask_top_k(chip, crise_sal, budget, invert=True), true_idx))

print("Done")

In [ ]:
# ---------------------------------------------------------------------------
# Results table
# ---------------------------------------------------------------------------

conditions = ["crise", "rise", "random", "bottom_crise"]
labels     = ["CRISE-guided", "RISE-guided", "Random", "Bottom-k CRISE (control)"]

rows = []
for budget in BUDGETS:
    row = {"Budget": f"{int(budget*100)}%"}
    for cond in conditions:
        vals = results[budget][cond]
        row[cond] = np.mean(vals) if vals else float("nan")
    rows.append(row)

res_df = pd.DataFrame(rows).set_index("Budget")
res_df.columns = labels
print(res_df.round(3).to_string())

In [ ]:
# ---------------------------------------------------------------------------
# Figure: rank-1 accuracy vs pixel budget — 4 conditions
# ---------------------------------------------------------------------------

colors    = ["#e74c3c", "#3498db", "#95a5a6", "#2ecc71"]
linestyle = ["-",       "--",       ":" ,       "-."]  

fig, ax = plt.subplots(figsize=(7, 4.5))

x = [int(b * 100) for b in BUDGETS]
for cond, label, color, ls in zip(conditions, labels, colors, linestyle):
    y = [np.mean(results[b][cond]) for b in BUDGETS]
    ax.plot(x, y, marker="o", label=label, color=color, linestyle=ls, linewidth=2)

ax.set_xlabel("Pixels masked (% of 112×112 chip)")
ax.set_ylabel("Rank-1 accuracy")
ax.set_title("CRISE-guided occlusion vs baselines")
ax.set_xticks(x)
ax.set_xticklabels([f"{v}%" for v in x])
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig_digital_validation.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig_digital_validation.png")

In [ ]:
# ---------------------------------------------------------------------------
# Visual example: one probe at all 4 conditions × 2 budgets (10% and 30%)
# ---------------------------------------------------------------------------

row = pairs.iloc[0]
img = cv2.imread(row["probe_path"])
chip = build_aligned_chip_112(img, app)
crise_sal = np.load(row["saliency_path_crise"]).astype(np.float32)
rise_sal  = np.load(row["saliency_path_rise"]).astype(np.float32)
rng_ex    = np.random.default_rng(0)

show_budgets = [0.10, 0.30]
fig, axes = plt.subplots(len(show_budgets) + 1, 4, figsize=(11, 3 * (len(show_budgets) + 1)))

# Row 0: saliency maps
axes[0, 0].imshow(cv2.cvtColor(chip, cv2.COLOR_BGR2RGB)); axes[0, 0].set_title("Original chip")
axes[0, 1].imshow(crise_sal, cmap="hot", vmin=0, vmax=1);  axes[0, 1].set_title("CRISE saliency")
axes[0, 2].imshow(rise_sal,  cmap="hot", vmin=0, vmax=1);  axes[0, 2].set_title("RISE saliency")
axes[0, 3].axis("off")

for row_i, b in enumerate(show_budgets):
    masked_chips = [
        mask_top_k(chip, crise_sal, b),
        mask_top_k(chip, rise_sal,  b),
        mask_random(chip, b, rng_ex),
        mask_top_k(chip, crise_sal, b, invert=True),
    ]
    for col_i, (mc, label) in enumerate(zip(masked_chips, labels)):
        axes[row_i + 1, col_i].imshow(cv2.cvtColor(mc, cv2.COLOR_BGR2RGB))
        axes[row_i + 1, col_i].set_title(f"{label}\n{int(b*100)}% masked", fontsize=8)

for ax in axes.ravel():
    ax.axis("off")

plt.suptitle(f"Occlusion example: {row['true_id']}", fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig_occlusion_example.png"), dpi=150, bbox_inches="tight")
plt.show()